# HMC-Resampled PDZ FLIP Datasets

Create `N` copies of the published PDZ FLIP split with `target` replaced by a joint HMC posterior draw of `log_r_mean`.

The published CSV contains fitness-like targets retained as `published_target`.

**Provisional overlap policy:** when a FLIP sequence occurs in both trans fits, each generated replicate randomly chooses either the trans-1 or trans-2 posterior value for that sequence. The selected fit is recorded in `hmc_source`. We should probably discuss more about this.

In [5]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "SciStanPy"))
from flipv3.flip_dsets import load_pdz_dataset

WT_PDZ = "GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQILSVNGVDLRNASHEQAAIALKNAGQTVTIIAQYKPEEYSRFEA"
COUNT_PATHS = {source: ROOT / "data" / f"{source}.tsv" for source in ("trans-1", "trans-2")}
HMC_PATHS = {
    
    "trans-1": ROOT / "amlt/large-clean/large-clean_82/pdz/trans-1/pdz_trans-1_exponential_exponential-20260407030507.nc",
    "trans-2": ROOT / "amlt/large-clean/large-clean_88/pdz/trans-2/pdz_trans-2_exponential_exponential_diagnosed.nc",
}
FLIP_PATH = ROOT / "data/single_to_double_from_flip_repo.csv"

In [6]:
def variant_map(source):
    """Map each SciStanPy variant index to its reconstructed FLIP PDZ:CRIPT sequence."""
    aa = pd.Series(load_pdz_dataset(str(COUNT_PATHS[source]))["variants"])
    pdz = aa.str[:50] + WT_PDZ[50:] if source == "trans-1" else WT_PDZ[:50] + aa.str[:50]
    sequence = (
        pdz.str.split("*", n=1, regex=False).str[0]
        + ":"
        + aa.str[-8:].str.split("*", n=1, regex=False).str[0]
    )
    return pd.DataFrame({"b": np.arange(len(aa)), "sequence": sequence})


flip = pd.read_csv(FLIP_PATH)
variant_maps = {source: variant_map(source) for source in COUNT_PATHS}
posteriors = {
    source: xr.open_dataset(path, group="posterior", engine="h5netcdf", chunks={})[
        "log_r_mean"
    ]
    for source, path in HMC_PATHS.items()
}

In [7]:
# Prove that each HMC b index has the same count row and variant order as SciStanPy.
alignment = []
for source, path in HMC_PATHS.items():
    counts = load_pdz_dataset(str(COUNT_PATHS[source]))
    with xr.open_dataset(path, group="observed_data", engine="h5netcdf") as observed:
        count_order_matches = (
            np.array_equal(observed["starting_counts"].values, counts["starting_counts"])
            and np.array_equal(observed["timepoint_counts"].values, counts["timepoint_counts"])
        )

    mapping = variant_maps[source]
    mapping_is_valid = (
        len(mapping) == posteriors[source].sizes["b"]
        and mapping["b"].equals(pd.Series(np.arange(len(mapping)), name="b"))
        and mapping["sequence"].is_unique
    )
    assert count_order_matches and mapping_is_valid

    # Attach the reconstructed sequence to b so the relationship is visible on the posterior.
    posteriors[source] = posteriors[source].assign_coords(
        sequence=("b", mapping["sequence"].to_numpy())
    )
    alignment.append(
        {
            "source": source,
            "HMC variants": posteriors[source].sizes["b"],
            "count order matches": count_order_matches,
            "sequence map valid": mapping_is_valid,
            "sequences in FLIP": mapping["sequence"].isin(flip["sequence"]).sum(),
        }
    )

pd.DataFrame(alignment)

,source,HMC variants,count order matches,sequence map valid,sequences in FLIP
0,trans-1,150769,True,True,581
1,trans-2,151624,True,True,308


In [ ]:
def make_hmc_flip_replicates(n, seed=23):
    """Return N FLIP DataFrames whose targets are replaced by joint HMC posterior draws."""
    rng = np.random.default_rng(seed)
    sampled = {}

    for source, posterior in posteriors.items():
        posterior = posterior.stack(sample=("chain", "draw")).transpose("sample", "b")
        rows = variant_maps[source].query("sequence in @flip.sequence").reset_index(drop=True)
        if n > posterior.sizes["sample"]:
            raise ValueError(f"Requested {n} datasets, but {source} has only {posterior.sizes['sample']} draws.")
        draw_ids = (
            np.arange(posterior.sizes["sample"])
            if n == posterior.sizes["sample"]
            else rng.choice(posterior.sizes["sample"], n, replace=False)
        )
        selected = posterior.isel(
            sample=draw_ids,
            b=xr.DataArray(rows["b"].to_numpy(), dims="variant"),
        )
        assert np.array_equal(selected["sequence"].values, rows["sequence"].to_numpy())
        sampled[source] = rows["sequence"], selected.compute().values

    base = flip.rename(columns={"target": "published_target"})
    replicates = []
    for replicate in range(n):
        candidates = pd.concat(
            [
                pd.DataFrame(
                    {
                        "sequence": sequences,
                        "target": values[replicate],
                        "hmc_source": source,
                    }
                )
                for source, (sequences, values) in sampled.items()
            ],
            ignore_index=True,
        )
        targets = candidates.groupby("sequence", group_keys=False).sample(
            n=1,
            random_state=int(rng.integers(2**32)),
        )
        dataset = base.merge(targets, on="sequence", validate="one_to_one")
        dataset.insert(0, "replicate", replicate)
        replicates.append(
            dataset[
                [
                    "replicate",
                    "sequence",
                    "target",
                    "hmc_source",
                    "set",
                    "validation",
                    "published_target",
                ]
            ]
        )

    return replicates

In [ ]:
MAX_HMC_DRAWS = min(
    posterior.sizes["chain"] * posterior.sizes["draw"]
    for posterior in posteriors.values()
)
N = 10  # set N = MAX_HMC_DRAWS to use every draw in natural stacked order
print(f"Maximum common HMC draws: {MAX_HMC_DRAWS:,}")

replicate_datasets = make_hmc_flip_replicates(N)
replicate_datasets[-1]

,replicate,sequence,target,hmc_source,set,validation,published_target
0,9,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-5.579584,trans-1,train,False,-0.834694
1,9,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,0.647648,trans-2,train,False,-0.085543
2,9,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,0.744121,trans-2,train,False,-0.057949
3,9,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,0.473193,trans-2,train,False,-0.164924
4,9,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,0.524232,trans-2,train,False,-0.150309
...,...,...,...,...,...,...,...
729,9,GEFDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-3.801759,trans-1,test,False,-0.464239
730,9,GSEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-1.520824,trans-1,test,False,-0.288265
731,9,GWEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-2.361767,trans-1,test,False,-0.497967
732,9,GCEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-2.864709,trans-1,test,False,-0.514674
